# Inventário de Estrutura dos Arquivos do BPS
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**  
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

Diagnostica a estrutura dos arquivos originais antes de qualquer tratamento: campos presentes em
cada ano, formato dos identificadores e cobertura de preenchimento. Foi esta rotina que revelou a
variação no formato do código de material entre os anos, condição para a definição da regra de
padronização aplicada na harmonização.

Também decide **quais anos entram no estudo**, pelo conteúdo de cada arquivo: só entram os arquivos que trazem
o código CATMAT do item e o CNPJ do fornecedor, as duas informações sem as quais não há item nem fornecedor para
calcular os indicadores. O resultado fica em `Auditoria/01_anos_do_estudo.csv`, lido pelas rotinas 02 e 03, que
só processam os anos aprovados aqui.

In [1]:
# pandas fixado em 2.2.3 (mesma serie 2.2; tem instalador pronto para o Python 3.13 do Colab): versoes diferentes alteram a leitura dos arquivos originais.
# No Colab, se for solicitado, reinicie a sessao apos a instalacao.
!pip install pandas==2.2.3 xlrd openpyxl --quiet
"""
Inventário de Estrutura dos Arquivos BPS
TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009-2023
Mônica Anatalia Bezerra de Araujo — MBA em Data Science e Analytics para Operações, POLI USP PRO

Objetivo:
    Ler todos os arquivos BPS (.xls e .xlsx) de uma pasta e gerar
    uma tabela onde cada linha é um arquivo e cada coluna indica
    se aquele campo existe (✓) ou não existe (—) naquele arquivo.

Como usar no Google Colab:
    1. Faça upload dos arquivos BPS para uma pasta no Google Drive
    2. Monte o Drive:
         from google.colab import drive
         drive.mount('/content/drive')
    3. Ajuste a variável PASTA_BPS com o caminho correto
    4. Execute o script
"""

import os
import pandas as pd
from pathlib import Path



In [2]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


In [3]:
# ─────────────────────────────────────────────
# CONFIGURAÇÃO — ajuste conforme seu ambiente
# ─────────────────────────────────────────────

# No Colab com Google Drive, use algo como:
PASTA_BPS = Path(PASTA_DADOS) / 'Base Tratamento Minimo'
ARQUIVO_SAIDA = 'inventario_estrutura_bps.xlsx'


In [4]:
# ─────────────────────────────────────────────
# FUNÇÕES AUXILIARES
# ─────────────────────────────────────────────

def detectar_header(caminho: str) -> int:
    """
    Arquivos BPS antigos (.xls) têm uma linha de aviso antes do cabeçalho.
    Esta função detecta automaticamente em qual linha está o cabeçalho real.
    """
    df_raw = pd.read_excel(caminho, header=None, nrows=5)
    for i, row in df_raw.iterrows():
        valores = [str(v) for v in row.values if pd.notna(v)]
        # Linha de cabeçalho real tem múltiplos valores não-nulos
        if len(valores) >= 3:
            return i
    return 0


def obter_colunas(caminho: str) -> list:
    """Retorna a lista de colunas de um arquivo BPS."""
    header_row = detectar_header(caminho)
    df = pd.read_excel(caminho, header=header_row, nrows=0)
    return list(df.columns)


def extrair_ano(nome_arquivo: str) -> str:
    """Extrai o ano do nome do arquivo (ex: BPS_2025.xlsx → 2025)."""
    import re
    match = re.search(r'(\d{4})', nome_arquivo)
    return match.group(1) if match else nome_arquivo



In [5]:
def gerar_inventario(pasta: str, saida: str):
    pasta = Path(pasta)

    # Coleta todos os arquivos BPS
    arquivos = sorted([
        f for f in pasta.iterdir()
        if f.suffix.lower() in ('.xls', '.xlsx')
        and 'BPS' in f.name.upper()
        and 'inventario' not in f.name.lower()
    ])

    if not arquivos:
        print(f"Nenhum arquivo BPS encontrado em: {pasta.resolve()}")
        return

    print(f"Arquivos encontrados: {len(arquivos)}")
    for f in arquivos:
        print(f"  • {f.name}")

    # Mapeia arquivo → colunas
    mapa = {}
    for arq in arquivos:
        try:
            colunas = obter_colunas(str(arq))
            mapa[arq.name] = colunas
            print(f"  ✓ {arq.name}: {len(colunas)} colunas")
        except Exception as e:
            mapa[arq.name] = []
            print(f"  ✗ {arq.name}: ERRO — {e}")

    # Universo de colunas únicas (ordem de aparição)
    todas_colunas = []
    for cols in mapa.values():
        for c in cols:
            if c not in todas_colunas:
                todas_colunas.append(c)

    # Monta a tabela de inventário
    linhas = []
    for nome_arq, colunas in mapa.items():
        linha = {'Arquivo': nome_arq, 'Ano': extrair_ano(nome_arq), 'Qtd Colunas': len(colunas)}
        for col in todas_colunas:
            linha[col] = '✓' if col in colunas else '—'
        linhas.append(linha)

    df_inventario = pd.DataFrame(linhas)
    df_inventario = df_inventario.sort_values('Ano').reset_index(drop=True)

    # ── Exporta para Excel ──────────────────────
    caminho_saida = pasta / saida
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df_inventario.to_excel(writer, sheet_name='Inventário', index=False)

        ws = writer.sheets['Inventário']

        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        # Cores
        fill_header   = PatternFill('solid', fgColor='1F4E79')   # azul escuro
        fill_presente = PatternFill('solid', fgColor='C6EFCE')   # verde claro
        fill_ausente  = PatternFill('solid', fgColor='FFCCCC')   # vermelho claro
        fill_meta     = PatternFill('solid', fgColor='BDD7EE')   # azul claro (metadados)

        fonte_header  = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        fonte_dados   = Font(name='Arial', size=9)
        borda_fina    = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'),  bottom=Side(style='thin')
        )

        n_cols = df_inventario.shape[1]
        n_rows = df_inventario.shape[0]

        for col_idx in range(1, n_cols + 1):
            letra = get_column_letter(col_idx)
            header_cell = ws[f'{letra}1']

            # Formata cabeçalho
            header_cell.fill      = fill_header
            header_cell.font      = fonte_header
            header_cell.alignment = Alignment(horizontal='center', wrap_text=True)
            header_cell.border    = borda_fina

            # Largura das colunas
            col_name = df_inventario.columns[col_idx - 1]
            if col_name in ('Arquivo', 'Ano', 'Qtd Colunas'):
                ws.column_dimensions[letra].width = 22
            else:
                ws.column_dimensions[letra].width = 28

            # Formata células de dados
            for row_idx in range(2, n_rows + 2):
                cell = ws[f'{letra}{row_idx}']
                valor = cell.value
                cell.font      = fonte_dados
                cell.alignment = Alignment(horizontal='center')
                cell.border    = borda_fina

                if col_name in ('Arquivo', 'Ano', 'Qtd Colunas'):
                    cell.fill = fill_meta
                elif valor == '✓':
                    cell.fill = fill_presente
                else:
                    cell.fill = fill_ausente

        # Congela primeira linha e primeiras 3 colunas
        ws.freeze_panes = 'D2'

    print(f"\nInventário salvo em: {caminho_saida.resolve()}")
    print(f"Total: {len(arquivos)} arquivos | {len(todas_colunas)} colunas únicas")
    return df_inventario

In [6]:
# ─────────────────────────────────────────────
# EXECUTA
# ─────────────────────────────────────────────
df = gerar_inventario(PASTA_BPS, ARQUIVO_SAIDA)
if df is not None:
    print("\nPrévia do resultado:")
    print(df[['Arquivo', 'Ano', 'Qtd Colunas']].to_string(index=False))

Arquivos encontrados: 24
  • BPS_2000.xls
  • BPS_2001.xls
  • BPS_2002.xls
  • BPS_2003.xls
  • BPS_2004.xls
  • BPS_2005.xls
  • BPS_2006.xls
  • BPS_2007.xls
  • BPS_2008.xls
  • BPS_2009.xls
  • BPS_2010.xls
  • BPS_2011.xls
  • BPS_2012.xls
  • BPS_2013.xls
  • BPS_2014.xls
  • BPS_2015.xlsx
  • BPS_2016.xlsx
  • BPS_2017.xlsx
  • BPS_2018.xlsx
  • BPS_2019.xlsx
  • BPS_2020.xlsx
  • BPS_2021.xlsx
  • BPS_2022.xlsx
  • BPS_2023.xlsx
  ✓ BPS_2000.xls: 11 colunas
  ✓ BPS_2001.xls: 11 colunas
  ✓ BPS_2002.xls: 14 colunas
  ✓ BPS_2003.xls: 11 colunas
  ✓ BPS_2004.xls: 11 colunas
  ✓ BPS_2005.xls: 11 colunas
  ✓ BPS_2006.xls: 11 colunas
  ✓ BPS_2007.xls: 11 colunas
  ✓ BPS_2008.xls: 11 colunas
  ✓ BPS_2009.xls: 22 colunas
  ✓ BPS_2010.xls: 22 colunas
  ✓ BPS_2011.xls: 22 colunas
  ✓ BPS_2012.xls: 22 colunas
  ✓ BPS_2013.xls: 27 colunas
  ✓ BPS_2014.xls: 27 colunas
  ✓ BPS_2015.xlsx: 27 colunas
  ✓ BPS_2016.xlsx: 24 colunas
  ✓ BPS_2017.xlsx: 22 colunas
  ✓ BPS_2018.xlsx: 20 colunas
  ✓

## Anos que entram no estudo
Regra pelo conteúdo: código CATMAT preenchido e CNPJ do fornecedor em pelo menos metade das linhas lidas.
Os anos fora da regra ficam na pasta de originais, mas não são convertidos nem harmonizados.

In [7]:
# ─────────────────────────────────────────────
# ANOS QUE ENTRAM NO ESTUDO (regra pelo conteúdo de cada arquivo)
# ─────────────────────────────────────────────
# Um arquivo entra se trouxer as duas informações sem as quais os indicadores não existem:
#   (1) código CATMAT do item: coluna "Código BR" preenchida;
#   (2) CNPJ do fornecedor: número de 11 a 14 dígitos numa coluna de fornecedor ("CNPJ Fornecedor" ou, em 2018–2020,
#       "Fornecedor"). Abaixo de 14 dígitos: a planilha guardou o CNPJ como número e perdeu os zeros à esquerda.
# A conferência lê as primeiras AMOSTRA linhas de cada arquivo. O ano final segue o período do estudo.
import re
import unicodedata

AMOSTRA = 1000
MIN_PREENCHIDO = 0.5          # metade das linhas da amostra
ANO_FIM_ESTUDO = 2023


def _norm(txt) -> str:
    txt = unicodedata.normalize('NFKD', str(txt)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', txt).strip().lower()


def conferir_arquivo(caminho: Path) -> dict:
    hr = detectar_header(str(caminho))
    df = pd.read_excel(caminho, header=hr, nrows=AMOSTRA, dtype=str)
    cols = {c: _norm(c) for c in df.columns}
    col_cod = [c for c, n in cols.items() if 'codigo br' in n or 'catmat' in n]
    col_forn = [c for c, n in cols.items() if 'fornecedor' in n]
    pct_cod = max((df[c].notna() & df[c].astype(str).str.strip().ne('')).mean() for c in col_cod) if col_cod else 0.0
    pct_cnpj, col_cnpj = 0.0, '—'
    for c in col_forn:
        dig = df[c].astype(str).str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True)
        p = dig.str.len().between(11, 14).mean()
        if p > pct_cnpj:
            pct_cnpj, col_cnpj = p, c
    ano = int(extrair_ano(caminho.name))
    motivos = []
    if pct_cod < MIN_PREENCHIDO:
        motivos.append('sem código CATMAT')
    if pct_cnpj < MIN_PREENCHIDO:
        motivos.append('sem CNPJ do fornecedor')
    if ano > ANO_FIM_ESTUDO:
        motivos.append(f'depois de {ANO_FIM_ESTUDO}')
    return {'ano': ano, 'arquivo': caminho.name,
            'coluna_codigo': ', '.join(map(str, col_cod)) or '—', 'pct_codigo_preenchido': round(pct_cod * 100, 1),
            'coluna_cnpj_fornecedor': str(col_cnpj), 'pct_cnpj_preenchido': round(pct_cnpj * 100, 1),
            'entra': 'Sim' if not motivos else 'Não', 'motivo': ' e '.join(motivos) if motivos else '—'}


arquivos = sorted(f for f in PASTA_BPS.iterdir()
                  if f.suffix.lower() in ('.xls', '.xlsx') and f.name.upper().startswith('BPS_'))
anos = pd.DataFrame([conferir_arquivo(f) for f in arquivos]).sort_values('ano').reset_index(drop=True)
pd.set_option('display.width', 200)
print(anos.drop(columns='coluna_codigo').to_string(index=False))

aprovados = anos.loc[anos.entra == 'Sim', 'ano'].tolist()
for motivo, g in anos[anos.entra == 'Não'].groupby('motivo'):
    print(f'Fora: {g.ano.min()} a {g.ano.max()} ({len(g)} arquivos): {motivo}')
print(f'Entram: {min(aprovados)} a {max(aprovados)} ({len(aprovados)} arquivos)')
assert aprovados == list(range(min(aprovados), max(aprovados) + 1)), 'anos aprovados com lacuna'

PASTA_AUDITORIA = Path(PASTA_DADOS) / 'Auditoria'
PASTA_AUDITORIA.mkdir(parents=True, exist_ok=True)
anos.to_csv(PASTA_AUDITORIA / '01_anos_do_estudo.csv', index=False, encoding='utf-8-sig')
print('Registro: Auditoria/01_anos_do_estudo.csv (lido pelas rotinas 02 e 03)')

 ano       arquivo  pct_codigo_preenchido coluna_cnpj_fornecedor  pct_cnpj_preenchido entra                                     motivo
2000  BPS_2000.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2001  BPS_2001.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2002  BPS_2002.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2003  BPS_2003.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2004  BPS_2004.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2005  BPS_2005.xls                    0.0                      —                  0.0   Não sem código CATMAT e sem CNPJ do fornecedor
2006  BPS_2006.xls                    0.0              